# Explore Azure Machine Learning workspace resources and assets

## Introduction

As a data scientist, you want to focus on training machine learning models. Ideally, you want to work with a service that gives you access to all the necessary infrastructure you need to train and deploy a model. You also want the service to allow you to track any work you do to make your model reproducible and robust.

Azure Machine Learning provides a platform for data scientists to train, deploy, and manage their machine learning models on the Microsoft Azure platform. Azure Machine Learning provides a comprehensive set of **resources** and **assets** to train and deploy effective machine learning models.

To use these resources and assets, you create an Azure Machine Learning workspace resource in your Azure subscription. In the Azure Machine Learning workspace, you can manage data, compute resources, models, endpoints, and other artifacts related to your machine learning workloads.

To get access to an Azure Machine Learning workspace, you first need to create the **Azure Machine Learning** service in your Azure subscription. The **workspace** is central place where you can work with all resources and assets available to train and deploy machine learning models. For **reproducibility**, the workspace stores a history of all training jobs, including logs, metrics, outputs, and a snapshot of your code.

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

In [3]:
subscription_id = "7068ddb3-fb1f-45a9-a02d-c8b87d8847b6"
resource_group = "rg-dp100-labs"
workspace_name = "mlw-dp100-labs"


ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace_name)

In [4]:
from azure.ai.ml import command

# configure job
job = command(
    code="./src",
    command="python train.py",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
    experiment_name="train-model"
)

# connect to workspace and submit job
returned_job = ml_client.create_or_update(job)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Invalid type ValidationException for a

MlException: {
  "result": "Failed",
  "errors": [
    {
      "message": "The filename, directory name, or volume label syntax is incorrect.; Not a valid URL.; In order to specify a git path, please provide the correct path prefixed with 'git+\n; In order to specify an existing codes, please provide the correct registry path prefixed with 'azureml://':\n; In order to specify an existing codes, please provide the correct registry path prefixed with 'azureml://':\n; Could not parse ./src. If providing an ARM id, it should start with a '/'.",
      "path": "component.code",
      "value": "azureml:./src"
    }
  ]
}

From the **Overview** page of the Azure Machine Learning workspace in the Azure portal, you can launch the Azure Machine Learning studio. The Azure Machine Learning studio is a web portal and provides an easy-to-use interface to create, manage, and use resources and assets in the workspace.

From the Azure portal, you can also give others access to the Azure Machine Learning workspace, using the **Access control**.

## Give access to the Azure Machine Learning workspace
You can give individual users or teams access to the Azure Machine Learning workspace. Access is granted in Azure using **role-based access control (RBAC)**, which you can configure in the **Access control** tab of the resource or resource group.

In the access control tab, you can manage permissions to restrict what actions certain users or teams can perform. For example, you could create a policy that only allows users in the *Azure administrators group* to create compute targets and datastores. While users in the *data scientists group* can create and run jobs to train models, and register models.

There are three general built-in roles that you can use across resources and resource groups to assign permissions to other users:

+ **Owner**: Gets full access to all resources, and can grant access to others using access control.
+ **Contributor**: Gets full access to all resources, but can't grant access to others.
+ **Reader**: Can only view the resource, but isn't allowed to make any changes.


Additionally, Azure Machine Learning has specific built-in roles you can use:

+ **AzureML Data Scientist**: Can perform all actions within the workspace, except for creating or deleting compute resources, or editing the workspace settings.
+ **AzureML Compute Operator**: Is allowed to create, change, and manage access the compute resources within a workspace.


Finally, if the built-in roles aren't meeting your needs, you can create a custom role to assign permissions to other users.



## Organize your workspaces
Initially, you might only work with one workspace. However, when working on large-scale projects, you might choose to use multiple workspaces.

You can use workspaces to group machine learning assets based on projects, deployment environments (for example, test and production), teams, or some other organizing principle.

# Identify Azure Machine Learning resources

Resources in Azure Machine Learning refer to the infrastructure you need to run a machine learning workflow. Ideally, you want someone like an administrator to create and manage the resources.

The resources in Azure Machine Learning include:

+ The workspace
+ Compute resources
+ Datastores

## Create and manage the workspace

The **workspace** is the top-level resource for Azure Machine Learning. Data scientists need access to the workspace to train and track models, and to deploy the models to endpoints.

However, you want to be careful with who has full access to the workspace. Next to references to compute resources and datastores, you can find all logs, metrics, outputs, models, and snapshots of your code in the workspace.

## Create and manage compute resources

One of the most important resources you need when training or deploying a model is compute. There are five types of compute in the Azure Machine Learning workspace:

+ **Compute instances:** Similar to a virtual machine in the cloud, managed by the workspace. Ideal to use as a **development environment** to run (Jupyter) notebooks.
+ **Compute clusters:** On-demand clusters of CPU or GPU compute nodes in the cloud, managed by the workspace. Ideal to use for **production workloads** as they automatically scale to your needs.
+ **Kubernetes clusters:** Allows you to create or attach an Azure Kubernetes Service (AKS) cluster. Ideal to **deploy trained machine learning models in production scenarios**.
+ **Attached computes:** Allows you to attach other Azure compute resources to the workspace, like Azure Databricks or Synapse Spark pools.
+ **Serverless compute:** A fully managed, on-demand compute you can use for **training jobs**.

 > *Note*: As Azure Machine Learning creates and manages serverless compute for you, it's not listed on the compute page in the studio. Learn more about how to use serverless compute for model training

Though compute is the most important resource when working with machine learning workloads, it can also be the most cost-intensive. Therefore, a best practice is to only allow administrators to create and manage compute resources. Data scientists shouldn't be allowed to edit compute, but only use the available compute to run their workloads.

## Create and manage datastores
The workspace doesn't store any data itself. Instead, all data is stored in datastores, which are references to Azure data services. The connection information to a data service that a datastore represents, is stored in the Azure Key Vault.

When a workspace is created, an Azure Storage account is created and automatically connected to the workspace. As a result, you have four datastores already added to your workspace:

+ `workspaceartifactstore`: Connects to the azureml container of the Azure Storage account created with the workspace. Used to store compute and experiment logs when running jobs.
+ `workspaceworkingdirectory`: Connects to the file share of the Azure Storage account created with the workspace used by the Notebooks section of the studio. Whenever you upload files or folders to access from a compute instance, the files or folders are uploaded to this file share.
+ `workspaceblobstore`: Connects to the Blob Storage of the Azure Storage account created with the workspace. Specifically the azureml-blobstore-... container. Set as the default datastore, which means that whenever you create a data asset and upload data, you store the data in this container.
+ `workspacefilestore`: Connects to the file share of the Azure Storage account created with the workspace. Specifically the azureml-filestore-... file share.

Additionally, you can create datastores to connect to other Azure data services. Most commonly, your datastores connects to an Azure Storage Account or Azure Data Lake Storage (Gen2) as those data services are most often used in data science projects.

# Identify Azure Machine Learning assets